### Logistic Regression with TF-IDF

Base model to benchmark the other models and compare their performance.

Make sure data is loaded and ready to be used:

In [2]:
from data_loader import build_and_save_split

build_and_save_split()

Split already exists at /Users/gavingirard/College/CS 4120/financial-sentiment-comparison/data/splits.parquet. Pass force=True to regenerate.


,sentence_id,text,label,agreement_tier,split
0,s00000,"According to Gran , the company has no plans t...",1,100,test
1,s00001,Technopolis plans to develop in stages an area...,1,66-74,train
2,s00002,The international electronic industry company ...,0,50-65,train
3,s00003,With the new production plant the company woul...,2,75-99,train
4,s00004,According to the company 's updated strategy f...,2,66-74,train
...,...,...,...,...,...
4833,s04833,LONDON MarketWatch -- Share prices ended lower...,0,100,val
4834,s04834,Rinkuskiai 's beer sales fell by 6.5 per cent ...,1,66-74,train
4835,s04835,Operating profit fell to EUR 35.4 mn from EUR ...,0,100,test
4836,s04836,Net sales of the Paper segment decreased to EU...,0,50-65,test


Set up TF-IDF and logistic regression pipeline:

In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from data_loader import load_split
from evaluation import evaluate_predictions

train_df, val_df, test_df = load_split()

train_text = train_df['text'].tolist()
val_text = val_df['text'].tolist()
test_text = test_df['text'].tolist()

train_labels = train_df['label'].tolist()
val_labels = val_df['label'].tolist()
test_labels = test_df['label'].tolist()

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_df=0.95, # Don't want features that are too common
        min_df=2 # Ignore super rare terms to get rid of noise
    )),
    ("classifier", LogisticRegression(
        penalty="l2", # L2 regularization for the model
        solver="liblinear", # Good for small datasets
        max_iter=10000 # Huge max iterations but I don't think it will need this long
    ))
])

# classifier__C targets the C parameter for skleanr's LogisticRegression which adjusts the regularization strength
params_to_try = {
    "classifier__C": [0.01, 0.1, 1, 10] # Different regulrization to try
}

Now we can use grid search to find the best hyperparameters for the model

In [17]:
grid_search = GridSearchCV(
    lr_pipeline,
    param_grid=params_to_try,
    cv=5,
    scoring="f1_macro", # F1 should work good for this, balanced between prescision and recall
)

grid_search.fit(train_text, train_labels)

print("Best regularization parameter: ", grid_search.best_params_)
print("Best cross-validation F1 score: ", grid_search.best_score_)

Best regularization parameter:  {'classifier__C': 10}
Best cross-validation F1 score:  0.5917695293825488


Save data to file for analysis (taken from logreg_tfidf_lml)

In [20]:
import pandas as pd
from pathlib import Path

Path("predictions").mkdir(exist_ok=True)

y_pred = grid_search.predict(test_text)
df_out = pd.DataFrame({
    "sentence_id": test_df["sentence_id"],
    "predicted_label": y_pred
})

output_path = "predictions/tfidf_logreg_predictions.csv"
df_out.to_csv(output_path, index=False)

print(f"Saved predictions to {output_path}")

Saved predictions to predictions/tfidf_logreg_predictions.csv


View result of how model performed

In [21]:
result = evaluate_predictions(output_path)

print(f"\nTest accuracy:  {result['overall']['accuracy']:.4f}")
print(f"Test macro-F1: {result['overall']['macro_f1']:.4f}")

print("\nBy agreement tier:")
for tier, m in result['by_tier'].items():
    print(f"  tier={tier:6s}  acc={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}  (n={m['n']})")


Test accuracy:  0.7727
Test macro-F1: 0.7211

By agreement tier:
  tier=100     acc=0.9071  macro_f1=0.8843  (n=226)
  tier=75-99   acc=0.7583  macro_f1=0.6395  (n=120)
  tier=66-74   acc=0.5526  macro_f1=0.3671  (n=76)
  tier=50-65   acc=0.5806  macro_f1=0.5475  (n=62)
